In [4]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional
import math

# Scaling Laws for Neural Language Models

## Experimental range

- **Model size:** 768 to 1.5 billion non-embedding parameters
- **Dataset size:** 22 million to 23 billion tokens
- **Architecture shape:** varied across depth, width, attention heads, and feed-forward dimension
- **Context length:** 1024 for most runs, with additional experiments on shorter contexts
- **Batch size:** $2^{19}$ for most runs, with separate experiments varying batch size to study the critical batch size

## Summary

Before this paper, the general belief was that larger language models tend to perform better. This paper makes that idea much more precise by showing that the test loss of autoregressive Transformer language models follows predictable power-law relationships with key scaling variables.

In particular, the paper studies how loss behaves as a function of:

- **model size** $N$
- **dataset size** $D$
- **optimally allocated compute** $C_{\min}$

when performance is mainly limited by one factor at a time.

The main contribution of the paper is not simply that “bigger models are better,” but that language-model performance scales in a **smooth, regular, and predictable** way. This allows the authors to fit empirical equations relating loss to scale and to use those equations to reason about training efficiency, overfitting, and compute allocation.

The paper reports several major findings:

1. **Performance scales strongly with model size**  
   Larger models achieve lower loss over a wide range of scales.

2. **Training curves show universality**  
   Models of different sizes tend to follow similar learning-curve shapes.

3. **Overfitting behaves systematically**  
   Overfitting depends in a predictable way on the relationship between model size and dataset size.

4. **Loss follows smooth power-law trends**  
   Improvements with scale are gradual and regular rather than abrupt.

5. **Larger models are more sample efficient**  
   Bigger models can make better use of each training example.

6. **Training to full convergence can be compute-inefficient**  
   Under a fixed compute budget, it can be better to train a larger model for fewer steps than to fully train a smaller one.

7. **There is an optimal batch-size regime**  
   Increasing batch size helps only up to a point, after which the gains become less useful.

Overall, the paper argues that language-model scaling is governed by empirical laws that relate loss to model size, dataset size, compute, and training progress. These laws make it possible to estimate trends in performance as models and training resources increase.

## Definitions of parameters

- $L$: cross-entropy loss of the language model
- $N$: number of **non-embedding** model parameters
- $D$: dataset size in **tokens**
- $C$: total training compute
- $C_{\min}$: optimally allocated compute required to reach a given loss
- $B$: batch size
- $S$: number of optimizer update steps
- $T$: total number of training tokens processed, where  
  $
  T = B \cdot S
  $

### Extra parameters used in the paper

- $E$: total number of training examples processed
- $E_{\min}$: minimum number of training examples needed to reach a given loss
- $S_{\min}$: minimum number of training steps needed to reach a given loss
- $B_{\text{crit}}$: critical batch size, the point beyond which increasing batch size gives much smaller efficiency gains

### Fitted constants in the scaling laws

- $N_c$: fitted scale constant for the model-size scaling law
- $D_c$: fitted scale constant for the dataset-size scaling law
- $C_c^{\min}$: fitted scale constant for the compute scaling law
- $S_c$: fitted scale constant for the step-based scaling law

### Scaling exponents

- $\alpha_N$: exponent for the model-size scaling law
- $\alpha_D$: exponent for the dataset-size scaling law
- $\alpha_C^{\min}$: exponent for the compute scaling law
- $\alpha_S$: exponent for the training-step scaling law
- $\alpha_B$: exponent related to batch-size scaling

## My project-specific notes

- In my project, $D$ is fixed, so the main relationship I care about is:
  $
  L(N \mid D=\text{fixed})
  $
- I also want to track:
  $
  C \approx 6NBS
  $
- and
  $
  T = BS
  $

This means my main logged quantities should be:

- model size $N$
- dataset size $D$
- batch size $B$
- number of steps $S$
- total tokens processed $T$
- compute estimate $C$
- train loss
- validation loss

##### Equation 1 — Loss as a function of model size

For models with a limited number of parameters, trained to convergence on sufficiently large datasets, the paper gives:

$
L(N) = \left(\frac{N_c}{N}\right)^{\alpha_N}
$

where:

- $L(N)$: test loss as a function of model size
- $N$: number of **non-embedding parameters**
- $N_c$: fitted scale constant for the model-size law
- $\alpha_N$: scaling exponent for model size

For the paper’s setup, the fitted values are:

$
\alpha_N \approx 0.076, \qquad N_c \approx 8.8 \times 10^{13}
$

### Interpretation

This equation says that when model size is the main bottleneck, the test loss decreases as model size increases, following a **power law**. The improvement is smooth but shows **diminishing returns**, since the exponent $\alpha_N$ is small.

### Important note

- $N_c$ is **not** the optimal model size.
- $N_c$ is a **fitted constant** that sets the horizontal position of the curve.
- If two experiments have the same $\alpha_N$ but different $N_c$, then on a **log-log plot** they will have the same slope but different intercepts.

### My project interpretation

In my project, the dataset is fixed, so this is one of the main equations I want to test:

$
L(N \mid D=\text{fixed})
$

This means I want to check whether validation loss in my setup also follows a power-law trend as model size increases.

### Matching function

A direct implementation of the paper-style equation is:

$
f_{\text{loss\_vs\_params}}(N; N_c, \alpha_N)=\left(\frac{N_c}{N}\right)^{\alpha_N}
$

A more flexible version for fitting my own experiments is:

$
f_{\text{loss\_vs\_params}}(N; A, \alpha, B)=A N^{-\alpha}+B
$

where $B$ allows the curve to flatten toward a loss floor.

In [7]:

# from the paper 
N_c = 8.8 * 10 ** -3
alpha_N = 0.076

def loss_vs_params_paper(N: float, N_c: float, alpha_N: float) -> float:
    """
    Kaplan Eq. style:
        L(N) = (N_c / N) ** alpha_N

    Parameters
    ----------
    N : float
        Number of non-embedding parameters.
    N_c : float
        Fitted scale constant for model-size scaling.
    alpha_N : float
        Model-size scaling exponent.
    """
    if N <= 0:
        raise ValueError("N must be > 0")
    return (N_c / N) ** alpha_N


def loss_vs_params_practical(N: int, A: float, alpha: float, B: float) -> float:
    """
    Practical fitted form:
        L(N) = A * N^(-alpha) + B

    Parameters
    ----------
    N : int
        Number of parameters.
    A : float
        Fitted scaling constant.
    alpha : float
        Fitted exponent.
    B : float
        Loss floor / offset.

    Returns
    -------
    float
        Predicted loss at model size N.
    """
    if N <= 0:
        raise ValueError("N must be > 0")
    return A * (N ** (-alpha)) + B

#### Equation 2 — Loss as a function of dataset size

For large models trained with a limited dataset and early stopping, the paper gives:

$
L(D) = \left(\frac{D_c}{D}\right)^{\alpha_D}
$

where:

- $L(D)$: test loss as a function of dataset size
- $D$: dataset size in **tokens**
- $D_c$: fitted scale constant for the dataset-size law
- $\alpha_D$: scaling exponent for dataset size

For the paper’s setup, the fitted values are:

$
\alpha_D \approx 0.095, \qquad D_c \approx 5.4 \times 10^{13} \text{ tokens}
$

### Interpretation

This equation says that when dataset size is the main bottleneck, the test loss decreases as the number of training tokens increases, following a **power law**. As with model size, the improvement is smooth but shows **diminishing returns**.

### Important note

- $D_c$ is **not** the required dataset size.
- $D_c$ is a **fitted constant** that sets the horizontal position of the curve.
- This law is most useful when dataset size is being varied while model size is large enough that data is the limiting factor.

### My project interpretation

In my project, the dataset is fixed, so I am **not** sweeping $D$ as the main experimental axis. This means I cannot directly fit a full loss-vs-dataset-size curve from my main sweep.

However, $D$ is still very important because it tells me how **data-limited** my setup may be.

So for my project:

$
D = \text{a measured constant}
$

and I should still calculate:

$
D_{\text{train}} = \text{total number of training tokens}
$

$
D_{\text{val}} = \text{total number of validation tokens}
$

This helps me interpret whether increases in model size are likely to keep helping or whether I am entering a data-limited regime.

### Matching function

A direct implementation of the dataset-scaling equation is:

$
f_{\text{loss\_vs\_data}}(D; D_c, \alpha_D)=\left(\frac{D_c}{D}\right)^{\alpha_D}
$

A more flexible version for fitting experiments is:

$
f_{\text{loss\_vs\_data}}(D; A_D, \alpha_D, B_D)=A_D D^{-\alpha_D}+B_D
$

where $B_D$ allows the curve to flatten toward a loss floor.

### Practical note for my notebook

Even though I may not fit this equation immediately, I still want it in the notebook because it is part of the full scaling-law framework, and it helps explain how fixed dataset size constrains my own experiments.

#### Equation 3 — Loss as a function of optimally allocated compute

When training with a limited amount of compute, a sufficiently large dataset, an optimally sized model, and a sufficiently small batch size, the paper gives:

$
L(C_{\min}) = \left(\frac{C_c^{\min}}{C_{\min}}\right)^{\alpha_C^{\min}}
$

where:

- $L(C_{\min})$: test loss as a function of optimally allocated compute
- $C_{\min}$: compute used under the paper’s optimal-allocation setting
- $C_c^{\min}$: fitted scale constant for the compute-scaling law
- $\alpha_C^{\min}$: scaling exponent for compute

For the paper’s setup, the fitted values are:

$
\alpha_C^{\min} \approx 0.050, \qquad C_c^{\min} \approx 3.1 \times 10^8 \text{ PF-days}
$

### Interpretation

This equation says that when compute is the main bottleneck, loss decreases as optimally allocated compute increases, following a **power law**.

However, the exponent is very small, which means performance improves **slowly** with compute. In other words, more compute helps, but with strong **diminishing returns**.

### Important note

- $C_{\min}$ does **not** mean any compute value.
- It refers to compute used under the paper’s **optimal allocation assumptions**, including:
  - sufficiently large dataset
  - optimally sized model
  - sufficiently small batch size
- $C_c^{\min}$ is a **fitted constant**, not a target compute budget.

### My project interpretation

In my project, I am **not** doing a full compute-optimal study yet.

This means I should be careful with this equation. I can keep it in the notebook as part of the full scaling-law framework, but for now I should mainly use compute as:

- a **logging variable**
- a **comparison variable** across runs

rather than my main fitted law.

So in my project, compute is more useful as:

$
C \approx 6NBS
$

or equivalently

$
C \approx 6NT
$

where:

- $N$: number of non-embedding parameters
- $B$: batch size in tokens
- $S$: number of optimizer steps
- $T = BS$: total tokens processed during training

### Matching function

A direct implementation of the compute-scaling equation is:

$
f_{\text{loss\_vs\_compute}}(C; C_c^{\min}, \alpha_C^{\min})=
\left(\frac{C_c^{\min}}{C}\right)^{\alpha_C^{\min}}
$

A more flexible fitting form is:

$
f_{\text{loss\_vs\_compute}}(C; A_C, \alpha_C, B_C)=A_C C^{-\alpha_C}+B_C
$

where $B_C$ allows the curve to flatten toward a loss floor.

### Practical note for my notebook

I may not fit this equation immediately, but I still want it in the notebook because it completes the three main scaling-law views:

- loss vs model size
- loss vs dataset size
- loss vs compute

For my current setup, it is mainly a **reference equation**, while the more directly useful compute equation is:

$
C \approx 6NBS
$

In [8]:
# from the paper 
alpha_C_min = 0.05
C_c_min = 3.1 * 10 ** 8  # PF-days in the paper

def loss_vs_compute_paper(C_min: float, C_c_min: float, alpha_C_min: float) -> float:
    if C_min <= 0:
        raise ValueError("C_min must be > 0")
    return (C_c_min / C_min) ** alpha_C_min

def training_compute(N: float, B: float, S: float) -> float:
    if N <= 0 or B <= 0 or S < 0:
        raise ValueError("N and B must be > 0, S must be >= 0")
    return 6 * N * B * S

#### Equation 4 — Critical batch size

The paper defines the **critical batch size** through the relationship between the number of training steps and the number of examples processed when training to a fixed target loss $L$: :contentReference[oaicite:0]{index=0}

$
\left(\frac{S}{S_{\min}} - 1\right)\left(\frac{E}{E_{\min}} - 1\right)=1
$

where:

- $S$: actual number of training steps
- $S_{\min}$: minimum number of steps needed to reach a target loss
- $E$: total number of training examples processed
- $E_{\min}$: minimum number of examples needed to reach the target loss

From this, the paper defines the critical batch size as:

$
B_{\text{crit}}(L) \equiv \frac{E_{\min}}{S_{\min}}
$

### Power-law fit for critical batch size

The paper then fits the critical batch size as a power law in the loss:

$
B_{\text{crit}}(L) \approx \frac{B_*}{L^{1/\alpha_B}}
$

with fitted values:

$
B_* \approx 2 \times 10^8, \qquad \alpha_B \approx 0.21
$

### Interpretation

This means that the ideal batch size mainly depends on the **current loss level**, not directly on model size. The paper argues that:

- for $B \lesssim B_{\text{crit}}$, increasing batch size gives little loss in compute efficiency
- for $B > B_{\text{crit}}$, increasing batch size gives **diminishing returns**
- training near $B \approx B_{\text{crit}}$ gives a roughly optimal tradeoff between time efficiency and compute efficiency. :contentReference[oaicite:3]{index=3}

The paper also states that training at the critical batch size requires about:

$
2S_{\min} \text{ training steps}
\qquad \text{and} \qquad
E = 2E_{\min} \text{ data examples}
$

for a roughly optimal time/compute compromise.

### My project interpretation

In my project, I do **not** think critical batch size should be the first equation I fit directly.

For me, batch size is mainly:

- a **stability choice**
- a **memory constraint**
- a **throughput choice**

So I should use this equation more as an **efficiency guideline** than a main scaling-law target.

In practice, this means:

- I should not assume bigger batch is always better
- I should expect there to be a point where increasing batch helps much less
- I may later test batch size separately with model size fixed

### Matching functions

A direct implementation of the paper-style fit is:

$
f_{\text{batch\_crit}}(L; B_*, \alpha_B)=\frac{B_*}{L^{1/\alpha_B}}
$

A simpler heuristic version for my own experiments is:

$
f_{\text{batch\_crit}}(L; k,p)=kL^{-p}
$

where $k$ and $p$ would be fitted from my own runs if I decide to study batch-size scaling later.

### Practical note for my notebook

I should include this equation because it connects batch size to training efficiency, but I should be careful not to overuse the paper's fitted constants in my own small-scale setup. For now, the main use of this section is to remind myself that batch size is part of the scaling-law framework and affects how efficiently training compute is used.

In [9]:
# from the paper
B_star = 2 * 10**8
alpha_B = 0.21

def b_crit_from_loss(L: float, B_star: float, alpha_B: float) -> float:
    """
    Kaplan paper-style critical batch size:
        B_crit(L) = B_star / L^(1 / alpha_B)

    Parameters
    ----------
    L : float
        Loss value.
    B_star : float
        Fitted batch-size scale constant.
    alpha_B : float
        Batch-size scaling exponent.

    Returns
    -------
    float
        Critical batch size.
    """
    if L <= 0:
        raise ValueError("L must be > 0")
    if alpha_B <= 0:
        raise ValueError("alpha_B must be > 0")
    return B_star / (L ** (1 / alpha_B))

def b_crit_from_emin_smin(E_min: float, S_min: float) -> float:
    """
    Definition of critical batch size:
        B_crit = E_min / S_min
    """
    if E_min <= 0:
        raise ValueError("E_min must be > 0")
    if S_min <= 0:
        raise ValueError("S_min must be > 0")
    return E_min / S_min

def s_min_from_batch(S: float, B: float, B_crit: float) -> float:
    """
    Effective minimum-step quantity:
        S_min(S) = S / (1 + B_crit / B)
    """
    if S <= 0:
        raise ValueError("S must be > 0")
    if B <= 0:
        raise ValueError("B must be > 0")
    if B_crit < 0:
        raise ValueError("B_crit must be >= 0")
    return S / (1 + B_crit / B)

L = 3.2
B_crit = b_crit_from_loss(L, B_star, alpha_B)
print(B_crit)

786237.7797952773


#### Equation 5 — Joint loss as a function of model size and dataset size

For finite model size and finite dataset size, with early stopping, the paper gives the joint scaling law:

$$
L(N, D)=\left[\left(\frac{N_c}{N}\right)^{\alpha_N/\alpha_D}+\frac{D_c}{D}\right]^{\alpha_D}
$$

where:

- $L(N,D)$: test loss as a function of both model size and dataset size
- $N$: number of non-embedding parameters
- $D$: dataset size in tokens
- $N_c$: fitted scale constant for the model-size component
- $D_c$: fitted scale constant for the dataset-size component
- $\alpha_N$: model-size scaling exponent
- $\alpha_D$: dataset-size scaling exponent

### Interpretation

This equation combines the effects of:

- limited model capacity
- limited dataset size

into one joint loss law.

It says that performance improves predictably when both $N$ and $D$ increase together, but if one is held fixed while the other grows, the improvement eventually slows down.

This is one of the key equations behind the paper’s discussion of overfitting and diminishing returns.

### Important note

This is not just Equation 1 plus Equation 2 added together directly.

The model-size term is raised to the power:

$$
\frac{\alpha_N}{\alpha_D}
$$

before the full expression is raised to $\alpha_D$.

This asymmetry is part of the paper’s fitted ansatz for finite-data overfitting behavior.

### My project interpretation

Because my dataset is fixed, the main object I care about is:

$$
L(N \mid D=\text{fixed})
$$

So I will not be fitting the full 2D surface immediately.

Instead, I will treat $D$ as a measured constant and study how loss changes with $N$ under that constraint.

This means Equation 5 is still conceptually important, because it explains why increasing model size alone may eventually enter a data-limited regime.

### Practical fitting form for my project

Since $D$ is fixed, I will probably fit a simpler 1D curve:

$$
L(N)=A N^{-\alpha}+B
$$

where:

- $A$: fitted scaling constant
- $\alpha$: fitted exponent
- $B$: loss floor / offset

This is more practical for my small-scale fixed-dataset experiments.

### Matching functions

Paper-style joint function:

$$
f_{\text{loss-vs-params-data}}(N,D;N_c,D_c,\alpha_N,\alpha_D)
=
\left[
\left(\frac{N_c}{N}\right)^{\alpha_N/\alpha_D}
+
\frac{D_c}{D}
\right]^{\alpha_D}
$$

My project-style fixed-dataset fitting function:

$$
f_{\text{loss-vs-params}}(N;A,\alpha,B)=AN^{-\alpha}+B
$$

### Why I still keep this equation in my notebook

Even though I may not fit it immediately, this equation is important because it links:

- model-size scaling
- dataset-size scaling
- overfitting behavior

and helps explain why scaling laws should be interpreted jointly rather than only through model size.

In [ ]:
def joint_loss_params_data_paper(
    N: float,
    D: float,
    N_c: float,
    D_c: float,
    alpha_N: float,
    alpha_D: float,
) -> float:
    """
    Kaplan paper-style joint scaling law:
        L(N, D) = [ (N_c / N)^(alpha_N / alpha_D) + (D_c / D) ]^alpha_D

    Parameters
    ----------
    N : float
        Number of non-embedding parameters.
    D : float
        Dataset size in tokens.
    N_c : float
        Fitted scale constant for model-size scaling.
    D_c : float
        Fitted scale constant for dataset-size scaling.
    alpha_N : float
        Model-size scaling exponent.
    alpha_D : float
        Dataset-size scaling exponent.

    Returns
    -------
    float
        Predicted loss from the joint model/data scaling law.
    """
    if N <= 0:
        raise ValueError("N must be > 0")
    if D <= 0:
        raise ValueError("D must be > 0")
    if alpha_D == 0:
        raise ValueError("alpha_D must be non-zero")

    return (((N_c / N) ** (alpha_N / alpha_D)) + (D_c / D)) ** alpha_D

def loss_vs_params_practical(N: float, A: float, alpha: float, B: float) -> float:
    if N <= 0:
        raise ValueError("N must be > 0")
    return A * (N ** (-alpha)) + B

#### Equation 6 — Joint loss as a function of model size and training steps

To model loss as a function of both model size and training progress in the infinite-data limit, the paper uses:

$$
L(N, S_{\min})=
\left(\frac{N_c}{N}\right)^{\alpha_N}
+
\left(\frac{S_c}{S_{\min}}\right)^{\alpha_S}
$$

where:

- $L(N,S_{\min})$: test loss as a function of model size and effective training steps
- $N$: number of non-embedding parameters
- $S_{\min}$: minimum number of steps needed to reach a target loss, adjusted to the critical batch-size setting
- $N_c$: fitted scale constant for the model-size term
- $S_c$: fitted scale constant for the training-step term
- $\alpha_N$: model-size scaling exponent
- $\alpha_S$: training-step scaling exponent

For the paper’s fit, the parameters are approximately:

$$
\alpha_N \approx 0.077, \qquad
\alpha_S \approx 0.76, \qquad
N_c \approx 6.5 \times 10^{13}, \qquad
S_c \approx 2.1 \times 10^3
$$

### Interpretation

This equation separates loss into two components:

- a **model-size term**, which reflects limits from finite model capacity
- a **training-step term**, which reflects limits from not training long enough

This is important because it helps distinguish two different reasons for high loss:

- the model may be too small
- the model may not have been trained for enough effective steps

### Why $S_{\min}$ is used instead of raw $S$

The paper does not use raw training steps directly. Instead, it defines:

$$
S_{\min}(S)=\frac{S}{1+B_{\text{crit}}(L)/B}
$$

where:

- $S$: actual number of optimizer steps
- $B$: batch size
- $B_{\text{crit}}(L)$: critical batch size at loss level $L$

This adjusts the observed step count to the equivalent minimum-step count at the critical batch-size setting.

### My project interpretation

This equation is very useful for my project because it tells me that loss depends not only on model size, but also on how far training has progressed.

So even if I keep dataset size fixed, I still need to think about:

- whether a large model is actually better
- or whether it simply has not been trained long enough

This makes Equation 6 one of the most important equations for interpreting scaling results in practice.

### Matching functions

Paper-style joint function:

$$
f_{\text{loss-vs-params-steps}}(N,S_{\min};N_c,S_c,\alpha_N,\alpha_S)
=
\left(\frac{N_c}{N}\right)^{\alpha_N}
+
\left(\frac{S_c}{S_{\min}}\right)^{\alpha_S}
$$

Step-adjustment function:

$$
f_{\text{Smin}}(S,B,B_{\text{crit}})
=
\frac{S}{1+B_{\text{crit}}/B}
$$

### Why I keep this equation in my notebook

This equation links:

- model size
- training progress
- batch-efficiency correction

and helps explain why poor scaling results can come from either insufficient model capacity or insufficient effective training time.

In [ ]:
def joint_loss_params_steps_paper(
    N: float,
    S_min: float,
    N_c: float,
    S_c: float,
    alpha_N: float,
    alpha_S: float,
) -> float:
    """
    Kaplan paper-style joint scaling law:
        L(N, S_min) = (N_c / N)^alpha_N + (S_c / S_min)^alpha_S

    Parameters
    ----------
    N : float
        Number of non-embedding parameters.
    S_min : float
        Effective minimum number of training steps.
    N_c : float
        Fitted scale constant for model-size scaling.
    S_c : float
        Fitted scale constant for step-based scaling.
    alpha_N : float
        Model-size scaling exponent.
    alpha_S : float
        Step-based scaling exponent.

    Returns
    -------
    float
        Predicted loss from the joint model-size/training-step law.
    """
    if N <= 0:
        raise ValueError("N must be > 0")
    if S_min <= 0:
        raise ValueError("S_min must be > 0")

    return (N_c / N) ** alpha_N + (S_c / S_min) ** alpha_S


def s_min_from_batch(S: float, B: float, B_crit: float) -> float:
    """
    Effective minimum-step quantity:
        S_min = S / (1 + B_crit / B)
    """
    if S <= 0:
        raise ValueError("S must be > 0")
    if B <= 0:
        raise ValueError("B must be > 0")
    if B_crit < 0:
        raise ValueError("B_crit must be >= 0")

    return S / (1 + B_crit / B)

#### Compute bridge — tokens, steps, and training compute

To connect model size, batch size, training steps, and compute, the paper uses a simple set of bridge equations.

### Total tokens processed

The total number of tokens processed during training is:

$$
T = B \cdot S
$$

where:

- $T$: total training tokens processed
- $B$: batch size in tokens
- $S$: number of optimizer steps

This is one of the most useful bookkeeping equations for my project.

### Approximate training compute

The paper approximates non-embedding training compute as:

$$
C \approx 6NBS
$$

Using $T = BS$, this can also be written as:

$$
C \approx 6NT
$$

where:

- $C$: total training compute
- $N$: number of non-embedding parameters
- $B$: batch size in tokens
- $S$: number of optimizer steps
- $T$: total number of tokens processed

### Interpretation

These equations connect the scaling-law variables to actual training runs.

They show that compute increases with:

- larger models
- larger batches
- more training steps
- more total tokens processed

For my project, this means I should log not only loss, but also:

- model size $N$
- batch size $B$
- training steps $S$
- total tokens processed $T$
- compute estimate $C$

### Compute with batch-efficiency correction

The paper also defines an effective compute quantity under the critical batch-size framework:

$$
C_{\min}(C)=\frac{C}{1+B/B_{\text{crit}}(L)}
$$

where:

- $C_{\min}(C)$: compute adjusted to the critical batch-size setting
- $C$: raw training compute
- $B$: batch size
- $B_{\text{crit}}(L)$: critical batch size at loss level $L$

### Interpretation of $C_{\min}$

This equation says that if batch size becomes too large relative to the critical batch size, then some of the compute is used inefficiently.

So:

- raw compute $C$ tells me how much total compute I used
- adjusted compute $C_{\min}$ tells me how much of that compute was effectively useful under the paper’s efficiency model

### My project interpretation

For my current setup, the most important equations are still:

$$
T = BS
$$

and

$$
C \approx 6NBS = 6NT
$$

These are the main equations I can directly use for logging and comparing runs.

The corrected compute equation

$$
C_{\min}(C)=\frac{C}{1+B/B_{\text{crit}}(L)}
$$

is still important conceptually, but I will probably treat it as a later refinement once I study batch-size effects more carefully.

### Matching functions

Total tokens processed:

$$
f_{\text{tokens}}(B,S)=BS
$$

Approximate training compute:

$$
f_{\text{compute}}(N,B,S)=6NBS
$$

Compute from total tokens:

$$
f_{\text{compute-from-tokens}}(N,T)=6NT
$$

Batch-corrected effective compute:

$$
f_{\text{compute-min}}(C,B,B_{\text{crit}})
=
\frac{C}{1+B/B_{\text{crit}}}
$$

### Why I keep this section in my notebook

This section is the bridge between:

- theoretical scaling-law equations
- actual training logs
- compute-aware model comparison

It lets me move from abstract variables like $N$, $S$, and $B$ to quantities I can actually measure in my own experiments.